# Notebook 3: Fine-tuning DNABERT-2 for Promoter Classification

**BMI 511 — Spring 2026**
**Instructor:** Pratik Dutta, Ph.D. | Department of Biomedical Informatics, Stony Brook University

---

## Learning objectives

1. Understand the **fine-tuning recipe**: pretrained encoder + classification head + supervised loss.
2. Prepare a real **promoter vs non-promoter** dataset with HuggingFace `datasets`.
3. Fine-tune DNABERT-2 with the HuggingFace `Trainer` API.
4. Evaluate with accuracy, MCC, F1, and confusion matrix.
5. Compare fine-tuned performance against the linear-probe baseline from Notebook 2.

> Runtime: **T4 GPU required**. Training takes ~10–15 min on the reduced dataset below.


## 1. The fine-tuning recipe

```
  DNA sequence
      │
      ▼
  Tokenizer  (BPE, DNABERT-2)
      │
      ▼
  DNABERT-2 encoder  ──────  pretrained, 117M params, starts initialized
      │
      ▼
  [CLS] hidden state  (768-d)
      │
      ▼
  Linear classifier head  ──  trained from scratch
      │
      ▼
  logits → softmax → label
```

During fine-tuning we update **both** the encoder weights and the new classification head using labeled examples. The pretrained weights act as a **warm start** that encodes general DNA knowledge; we only need a small labeled dataset to adapt.

## 2. Setup

In [ ]:
!pip install -q transformers==4.44.2 datasets==2.21.0 accelerate==0.34.0 einops scikit-learn

In [ ]:
import os, random
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import (
    AutoTokenizer,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)
from datasets import Dataset
from sklearn.metrics import (
    accuracy_score, f1_score, matthews_corrcoef,
    confusion_matrix, classification_report,
)

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

sns.set_style("whitegrid")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
assert device == "cuda", "Please switch to a GPU runtime (Runtime > Change runtime type > T4 GPU)."

## 3. Dataset — synthetic promoter-like classification

For this teaching notebook we build a clean, self-contained binary dataset:

* **Positive (label = 1)**: promoter-like sequences with a **TATA box** (`TATAAA`) in the -25 region plus slightly higher GC content.
* **Negative (label = 0)**: random sequences with uniform nucleotide composition.

This is intentionally an easy task so students can see the full fine-tuning loop run to convergence in ~10 minutes. In a research setting you would instead load the **GUE promoter benchmark** or your own ChIP-seq / FANTOM CAGE data with the same code below — just swap the `Dataset.from_dict(...)` call for `load_dataset("InstaDeepAI/genomic-benchmarks", ...)` or a local CSV.


In [ ]:
def make_promoter_dataset(n_per_class=400, length=200, seed=0):
    rng = np.random.default_rng(seed)
    seqs, labels = [], []

    # Positives: promoter-like with TATA box
    for _ in range(n_per_class):
        bg = ''.join(rng.choice(list("ACGT"), size=length, p=[0.22, 0.28, 0.28, 0.22]))
        # insert TATAAA around position length - 25 (the canonical TATA position)
        pos = length - 25 + rng.integers(-3, 4)
        seq = bg[:pos] + "TATAAA" + bg[pos + 6:]
        seqs.append(seq)
        labels.append(1)

    # Negatives: random uniform DNA
    for _ in range(n_per_class):
        seq = ''.join(rng.choice(list("ACGT"), size=length))
        seqs.append(seq)
        labels.append(0)

    return seqs, labels

seqs, labels = make_promoter_dataset(n_per_class=400, length=200, seed=SEED)
print(f"Total sequences: {len(seqs)}   positives: {sum(labels)}   negatives: {len(labels) - sum(labels)}")
print(f"Example positive: ...{seqs[0][-40:]}")
print(f"Example negative: ...{seqs[-1][-40:]}")

In [ ]:
# Split into train / validation / test (70 / 15 / 15)
from sklearn.model_selection import train_test_split

X_tmp, X_test, y_tmp, y_test = train_test_split(seqs, labels, test_size=0.15, stratify=labels, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=0.1765, stratify=y_tmp, random_state=SEED)

print(f"train {len(X_train)}   val {len(X_val)}   test {len(X_test)}")

train_ds = Dataset.from_dict({"sequence": X_train, "label": y_train})
val_ds   = Dataset.from_dict({"sequence": X_val,   "label": y_val})
test_ds  = Dataset.from_dict({"sequence": X_test,  "label": y_test})

## 4. Tokenize

In [ ]:
MODEL_NAME = "zhihan1996/DNABERT-2-117M"
tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

def tokenize_fn(batch):
    return tok(batch["sequence"], truncation=True, max_length=256)

train_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=["sequence"])
val_ds   = val_ds.map(tokenize_fn,   batched=True, remove_columns=["sequence"])
test_ds  = test_ds.map(tokenize_fn,  batched=True, remove_columns=["sequence"])

collator = DataCollatorWithPadding(tokenizer=tok)
print("Tokenization done. A training example:")
print(train_ds[0])

## 5. Model with a classification head

In [ ]:
from transformers import BertForSequenceClassification, BertConfig

config = BertConfig.from_pretrained(MODEL_NAME, num_labels=2)
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    config=config,
).to(device)

n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params     : {n_total/1e6:.1f} M")
print(f"Trainable params : {n_train/1e6:.1f} M   (full fine-tune)")

## 6. Training configuration

Key hyperparameters for fine-tuning encoder models on small DNA datasets:

* **Learning rate**: `1e-5` to `5e-5` (lower than classical DL, because we're adapting a pretrained model).
* **Batch size**: as large as GPU memory allows; 16–32 on a T4.
* **Epochs**: 3–5 is usually enough; more tends to overfit.
* **Weight decay**: 0.01 is a safe default.


In [ ]:
def compute_metrics(eval_pred):
    logits, labels_ = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels_, preds),
        "f1":       f1_score(labels_, preds, average="binary"),
        "mcc":      matthews_corrcoef(labels_, preds),
    }

training_args = TrainingArguments(
    output_dir="./dnabert2_promoter",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="mcc",
    logging_steps=20,
    report_to="none",
    seed=SEED,
    fp16=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tok,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

## 7. Train

In [ ]:
train_result = trainer.train()
print("\nFinal training metrics:")
for k, v in train_result.metrics.items():
    print(f"  {k:30s}: {v}")

## 8. Evaluate on the held-out test set

In [ ]:
test_metrics = trainer.evaluate(test_ds)
print("Test metrics:")
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f"  {k:30s}: {v:.4f}")

In [ ]:
# Predictions and confusion matrix
pred_out = trainer.predict(test_ds)
y_pred = np.argmax(pred_out.predictions, axis=1)
y_true = np.array(pred_out.label_ids)

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["non-promoter", "promoter"],
            yticklabels=["non-promoter", "promoter"])
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Confusion matrix (test)")
plt.tight_layout(); plt.show()

print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=["non-promoter", "promoter"]))

## 9. Compare to a linear probe baseline

Is fine-tuning worth the compute? Let's run the same dataset through **frozen embeddings + logistic regression** and compare.


In [ ]:
from transformers import BertModel, BertConfig
from sklearn.linear_model import LogisticRegression

config_base = BertConfig.from_pretrained(MODEL_NAME)
base = BertModel.from_pretrained(
    MODEL_NAME,
    config=config_base,
).to(device)
base.eval()

@torch.no_grad()
def embed(seqs, batch_size=32):
    out_vecs = []
    for i in range(0, len(seqs), batch_size):
        batch = seqs[i:i+batch_size]
        enc = tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=256).to(device)
        h = base(**enc)
        h = h[0] if isinstance(h, tuple) else h.last_hidden_state
        mask = enc["attention_mask"].unsqueeze(-1).float()
        pooled = (h * mask).sum(1) / mask.sum(1).clamp(min=1)
        out_vecs.append(pooled.cpu().numpy())
    return np.concatenate(out_vecs, 0)

E_train = embed(X_train)
E_test  = embed(X_test)

probe = LogisticRegression(max_iter=2000, C=1.0).fit(E_train, y_train)
probe_pred = probe.predict(E_test)

print(f"Linear probe    : acc = {accuracy_score(y_test, probe_pred):.3f}   "
      f"MCC = {matthews_corrcoef(y_test, probe_pred):.3f}")
print(f"Fine-tuned      : acc = {test_metrics['eval_accuracy']:.3f}   "
      f"MCC = {test_metrics['eval_mcc']:.3f}")

**Interpretation.** On an easy task, the linear probe often comes very close to the fine-tuned model. On harder tasks (e.g. tissue-specific enhancers, splice sites with subtle motifs) the gap between linear probe and full fine-tuning widens substantially. That gap is the empirical justification for full fine-tuning.

## 10. Exercises

1. Reduce `num_train_epochs` to 1. How much does test MCC drop?
2. Set `learning_rate=1e-4`. Does training become unstable? Why might it?
3. Replace `AutoModelForSequenceClassification` training with **LoRA adapters** (`peft` library) and compare parameter count vs accuracy.
4. Swap the synthetic dataset for the **GUE promoter benchmark**:
   ```python
   from datasets import load_dataset
   ds = load_dataset("InstaDeepAI/nucleotide_transformer_downstream_tasks", "promoter_all")
   ```

## 11. Recap

* Fine-tuning = pretrained encoder + new classifier head + small labeled dataset.
* The HuggingFace `Trainer` API handles the training loop, evaluation, checkpointing, and metric tracking.
* For DNA tasks, use `learning_rate ~ 3e-5`, `epochs 3–5`, `fp16=True` on a T4 GPU.
* Always compare full fine-tuning to a **linear probe on frozen embeddings** — it's your strongest free baseline.

Next: **Notebook 4** will use the fine-tuned or pretrained model to predict the **functional impact of single-nucleotide variants**, the core idea behind DeepVRegulome.